# PRODCOM Product Classification with OpenAI

This notebook demonstrates how to classify products using the PRODCOM classification system powered by OpenAI's language models.

## Requirements:
- An OpenAI API Key (you'll be prompted to enter it)
- Python 3.8+
- The `prodcom_2023_classification.csv` file

## How it works:
1. The system takes product information (name, description, URL)
2. Navigates through the PRODCOM classification hierarchy (Section → Division → Group → Class → CPA5 → CPA6 → PRODCOM)
3. Uses OpenAI to select the best option at each level
4. Returns the final PRODCOM code and description

## Step 1: Install Dependencies

In [ ]:
# Install required packages
import subprocess
import sys

packages = ['openai', 'pandas', 'python-dotenv']

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} installed successfully")

## Step 2: Import Libraries

In [ ]:
import os
import re
import pandas as pd
import logging
import time
import random
from datetime import datetime
from openai import OpenAI
from getpass import getpass

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ All libraries imported successfully")

## Step 3: Setup Configuration and API Key

In [ ]:
# Configuration
class Config:
    """Configuration settings for the classification system"""
    OPENAI_MODEL = "gpt-4o-mini"
    TEMPERATURE = 0.0
    SLEEP_BETWEEN_PRODUCTS = 0.1  # seconds
    MAX_RETRIES = 5
    
    # File paths - adjust as needed
    PRODCOM_FILE = "./input/prodcom_2023_classification.csv"
    OUTPUT_DIR = "./output"
    OUTPUT_FILE_NAME = "prodotti_classificati_notebook"

config = Config()

# Get OpenAI API Key
api_key = getpass("Enter your OpenAI API Key: ")
if not api_key or api_key.strip() == "":
    raise ValueError("OpenAI API Key is required!")

client = OpenAI(api_key=api_key)
print("✓ OpenAI client initialized successfully")

## Step 4: Load PRODCOM Classification Data

In [ ]:
# Load PRODCOM classification data
try:
    prodcom_df = pd.read_csv(
        config.PRODCOM_FILE,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        delimiter="\t"
    )
    logger.info(f"✓ Loaded {len(prodcom_df)} PRODCOM records")
    print(f"✓ PRODCOM data loaded: {len(prodcom_df)} records")
except FileNotFoundError:
    logger.error(f"File not found: {config.PRODCOM_FILE}")
    print(f"Error: Cannot find {config.PRODCOM_FILE}")
    print("Please ensure the file is in the correct location or update the path in the configuration.")
    raise

# Display sample data
print("\nSample PRODCOM data:")
print(prodcom_df.head(3))

## Step 5: Define Helper Functions

In [ ]:
def clean_string(s: str) -> str:
    """
    Remove tabs, newlines, multiple spaces, and trim whitespace.
    """
    # Remove tabs and newlines
    s = re.sub(r"[\t\n\r]+", " ", s)
    # Replace multiple spaces with single space
    s = re.sub(r" +", " ", s)
    # Remove leading/trailing spaces
    s = s.strip()
    return s


def get_sections():
    """Extract unique sections from PRODCOM data"""
    sections_df = prodcom_df.drop_duplicates(subset=["section"]).reset_index(drop=True)
    sections_df = sections_df[["section", "section_descr"]]
    my_dict = dict(zip(sections_df.iloc[:, 0], sections_df.iloc[:, 1]))
    return my_dict


def get_divisions_by_section(section):
    """Get divisions for a given section"""
    division_df = prodcom_df[prodcom_df["section"] == str(section)]
    division_df = division_df[["division", "division_descr"]]
    division_df = division_df.drop_duplicates(subset=["division"]).reset_index(drop=True)
    my_dict = dict(zip(division_df.iloc[:, 0], division_df.iloc[:, 1]))
    return my_dict


def get_groups_by_division(division):
    """Get groups for a given division"""
    group_df = prodcom_df[prodcom_df["division"] == str(division)]
    group_df = group_df[["group", "group_descr"]]
    group_df = group_df.drop_duplicates(subset=["group"]).reset_index(drop=True)
    my_dict = dict(zip(group_df.iloc[:, 0], group_df.iloc[:, 1]))
    return my_dict


def get_classes_by_group(group):
    """Get classes for a given group"""
    class_df = prodcom_df[prodcom_df["group"] == str(group)]
    class_df = class_df[["class", "class_descr"]]
    class_df = class_df.drop_duplicates(subset=["class"]).reset_index(drop=True)
    my_dict = dict(zip(class_df.iloc[:, 0], class_df.iloc[:, 1]))
    return my_dict


def get_cpa5_by_class(classe):
    """Get CPA5 codes for a given class"""
    cpa5_df = prodcom_df[prodcom_df["class"] == str(classe)]
    cpa5_df = cpa5_df[["cpa5", "cpa5_descr"]]
    cpa5_df = cpa5_df.drop_duplicates(subset=["cpa5"]).reset_index(drop=True)
    my_dict = dict(zip(cpa5_df.iloc[:, 0], cpa5_df.iloc[:, 1]))
    return my_dict


def get_cpa6_by_cpa5(cpa5):
    """Get CPA6 codes for a given CPA5"""
    cpa6_df = prodcom_df[prodcom_df["cpa5"] == str(cpa5)]
    cpa6_df = cpa6_df[["cpa6", "cpa6_descr"]]
    cpa6_df = cpa6_df.drop_duplicates(subset=["cpa6"]).reset_index(drop=True)
    my_dict = dict(zip(cpa6_df.iloc[:, 0], cpa6_df.iloc[:, 1]))
    return my_dict


def get_prodcoms_by_cpa6(cpa6):
    """Get PRODCOM codes for a given CPA6"""
    prodcoms_df = prodcom_df[prodcom_df["cpa6"] == str(cpa6)]
    prodcoms_df = prodcoms_df[["prodcom", "prodcom_descr"]]
    prodcoms_df = prodcoms_df.drop_duplicates(subset=["prodcom"]).reset_index(drop=True)
    my_dict = dict(zip(prodcoms_df.iloc[:, 0], prodcoms_df.iloc[:, 1]))
    return my_dict


def get_prodcom_description_by_code(prodcom_code):
    """Get description for a PRODCOM code"""
    prodcom_descr_by_code = prodcom_df.loc[prodcom_df["prodcom"] == str(prodcom_code), "prodcom_descr"].values
    try:
        return prodcom_descr_by_code[0]
    except IndexError:
        logger.warning(f"WARNING: PRODCOM code {prodcom_code} not found")
        return ""


print("✓ Helper functions defined")

## Step 6: Define OpenAI Classification Functions

In [ ]:
def invoke_openai_api_with_rate_limit(user_prompt, model=None, temperature=None, max_retries=5):
    """
    Call OpenAI API with rate limit handling and exponential backoff.
    """
    if model is None:
        model = config.OPENAI_MODEL
    if temperature is None:
        temperature = config.TEMPERATURE

    retries = 0
    backoff = 1

    while True:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "system",
                        "content": "You are an expert assistant specialized in classifying products using the PRODCOM classification system. Respond with only the code requested, nothing else."
                    },
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ],
                temperature=temperature,
                seed=42
            )
            
            result = response.choices[0].message.content
            logger.info(f"API call successful - Tokens used: {response.usage.total_tokens}")
            return result

        except Exception as e:
            if "429" in str(e) and retries < max_retries:
                wait_time = backoff + random.uniform(0, 0.5)
                logger.info(f"Rate limit hit. Waiting {wait_time:.2f}s before retry...")
                time.sleep(wait_time)
                backoff = backoff * 2
                retries = retries + 1
            else:
                logger.error(f"API Error: {e}")
                return f"ERROR: {e}"


def get_llm_best_selection(options: dict, product_description: str):
    """
    Use OpenAI to select the best classification option for a product.
    """
    if len(options) == 0:
        logger.warning("No options available for selection")
        return None
    
    if len(options) == 1:
        key = next(iter(options))
        return str(key)

    # Prepare prompt
    option_lines = [f"{code}: {description}" for code, description in list(options.items())[:30]]
    prompt = f"""Consider the following product description:
{product_description}

Select the most appropriate classification code from the following options:
"""
    prompt += "\n".join(option_lines)
    prompt += "\n\nReturn ONLY the code (no explanation):"

    best_selection = invoke_openai_api_with_rate_limit(prompt)
    return best_selection.strip()


def classify_product(product_description: str):
    """
    Classify a product by navigating through the PRODCOM hierarchy.
    Returns: (code, description, code_type, code_parent)
    """
    logger.info(f"Classifying product: {product_description[:100]}...")
    
    try:
        # Navigate through hierarchy: Section → Division → Group → Class → CPA5 → CPA6 → PRODCOM
        sections = get_sections()
        guessed_section = get_llm_best_selection(sections, product_description)
        logger.info(f"Selected section: {guessed_section}")

        divisions = get_divisions_by_section(guessed_section)
        guessed_division = get_llm_best_selection(divisions, product_description)
        logger.info(f"Selected division: {guessed_division}")

        groups = get_groups_by_division(guessed_division)
        guessed_group = get_llm_best_selection(groups, product_description)
        logger.info(f"Selected group: {guessed_group}")

        classes = get_classes_by_group(guessed_group)
        guessed_class = get_llm_best_selection(classes, product_description)
        logger.info(f"Selected class: {guessed_class}")

        cpa5s = get_cpa5_by_class(guessed_class)
        guessed_cpa5 = get_llm_best_selection(cpa5s, product_description)
        logger.info(f"Selected CPA5: {guessed_cpa5}")

        cpa6s = get_cpa6_by_cpa5(guessed_cpa5)
        guessed_cpa6 = get_llm_best_selection(cpa6s, product_description)
        logger.info(f"Selected CPA6: {guessed_cpa6}")

        prodcoms = get_prodcoms_by_cpa6(guessed_cpa6)
        guessed_prodcom = get_llm_best_selection(prodcoms, product_description)
        logger.info(f"Selected PRODCOM: {guessed_prodcom}")

        code_description = get_prodcom_description_by_code(guessed_prodcom)
        
        logger.info(f"Final classification - Code: {guessed_prodcom}, Description: {code_description}")
        
        return guessed_prodcom, code_description, "prodcom", "0"
    
    except Exception as e:
        logger.error(f"Error during classification: {e}")
        return "ERROR", str(e), "error", "0"


print("✓ OpenAI classification functions defined")

## Step 7: Create Sample Products

In [ ]:
# Create sample products for classification
sample_products = [
    {
        "name": "Fresh Pasta - Ravioli with Ricotta",
        "description": "Delicate ravioli pasta filled with creamy ricotta cheese and fresh herbs. Handmade using traditional Italian methods with premium wheat flour.",
        "id": "1",
        "domain": "example.com",
        "product_url": "https://example.com/pasta/ravioli",
        "product_img_url": "",
        "source_file": "sample_data.csv"
    },
    {
        "name": "Chocolate Chip Cookies",
        "description": "Crunchy cookies filled with Belgian chocolate chips. Made with butter and natural ingredients. Perfect for dessert or snacking.",
        "id": "2",
        "domain": "example.com",
        "product_url": "https://example.com/cookies/chocolate",
        "product_img_url": "",
        "source_file": "sample_data.csv"
    },
    {
        "name": "Extra Virgin Olive Oil",
        "description": "Premium Italian extra virgin olive oil from the Tuscany region. Cold-pressed, fruity flavor, ideal for salads and cooking. 500ml bottle.",
        "id": "3",
,
domain": "example.com",
        "product_url": "https://example.com/oils/olive",
        "product_img_url": "",
        "source_file": "sample_data.csv"
    },
    {
        "name": "Aged Parmesan Cheese",
        "description": "Aged Parmigiano Reggiano 24 months. Hard cheese with complex flavor. Produced in Emilia-Romagna region using traditional methods.",
        "id": "4",
        "domain": "example.com",
        "product_url": "https://example.com/cheese/parmesan",
        "product_img_url": "",
        "source_file": "sample_data.csv"
    },
    {
        "name": "Red Wine - Barolo",
        "description": "Dry red wine from Piedmont, Italy. Made from Nebbiolo grapes. Full-bodied with complex tannins. Perfect with pasta and meat dishes.",
        "id": "5",
        "domain": "example.com",
        "product_url": "https://example.com/wine/barolo",
        "product_img_url": "",
        "source_file": "sample_data.csv"
    }
]

products_df = pd.DataFrame(sample_products)
print("Sample products to be classified:")
print(products_df[["id", "name", "description"]].to_string())

## Step 8: Classify Products

In [ ]:
# Classify products
print("\n" + "="*80)
print("Starting product classification...")
print("="*80 + "\n")

results = []
total_products = len(products_df)

for idx, (_, row) in enumerate(products_df.iterrows(), start=1):
    print(f"\n[{idx}/{total_products}] Processing: {row['name']}")
    print("-" * 80)
    
    # Create product description for classification
    product_description = row['description']
    if len(product_description.strip()) < 50:
        # If description is too short, combine with name
        product_description = f"{row['name']}. {row['description']}"
    
    # Classify
    code, code_description, code_type, code_parent = classify_product(product_description)
    
    # Store result
    result = {
        "name": row["name"],
        "description": row["description"],
        "id": row["id"],
        "domain": row["domain"],
        "product_url": row["product_url"],
        "product_img_url": row["product_img_url"],
        "source_file": row["source_file"],
        "code_type": code_type,
        "code_parent": code_parent,
        "code": code,
        "code_description": code_description
    }
    results.append(result)
    
    print(f"✓ Classification Result:")
    print(f"  Code: {code}")
    print(f"  Description: {code_description[:100]}...")
    
    # Sleep between products to avoid rate limiting
    if idx < total_products:
        time.sleep(config.SLEEP_BETWEEN_PRODUCTS)

print("\n" + "="*80)
print("Classification completed!")
print("="*80)

## Step 9: Display and Save Results

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results)

print("\nClassification Results:")
print(results_df[["name", "code", "code_description"]].to_string())

# Prepare output file path
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")
output_filename = f"{config.OUTPUT_FILE_NAME}_{timestamp}.csv"
output_filepath = os.path.join(config.OUTPUT_DIR, output_filename)

# Save to CSV
try:
    # Prepare columns in the desired order
    output_columns = [
        "name", "description", "id", "domain", "product_url", "product_img_url",
        "source_file", "code_type", "code_parent", "code", "code_description"
    ]
    results_df[output_columns].to_csv(output_filepath, sep="\t", index=False, encoding="utf-8")
    print(f"\n✓ Results saved to: {output_filepath}")
except Exception as e:
    print(f"Error saving results: {e}")
    print("Results dataframe:")
    print(results_df)

## Summary

This notebook successfully:
1. ✓ Loaded the PRODCOM classification hierarchy
2. ✓ Classified sample products using OpenAI's language model
3. ✓ Navigated through the PRODCOM hierarchy automatically
4. ✓ Generated output CSV file with classifications

### To use with your own products:
- Replace the `sample_products` list with your own product data
- Ensure each product has at least `name` and `description` fields
- Re-run the classification cells

### Notes:
- Classification accuracy depends on product description quality
- API costs depend on the number of products and the OPENAI_MODEL used
- The `gpt-4o-mini` model offers a good balance between speed and cost
- Temperature is set to 0.0 for deterministic results